# DRISHTI-SSS YOLOv11 Model Training on Google Colab (SIH 2026)
This notebook trains **YOLOv11** on the **DRISHTI-SSS** side-scan sonar dataset using a free T4 GPU on Google Colab, evaluates on the test split, and exports `best.pt`, `metrics.json`, and confusion matrix plots to Google Drive.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# Create Output Directory in Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/AquaGuard_ML'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"✓ Output Directory Ready: {DRIVE_OUTPUT}")

In [ ]:
# Step 2: Install Ultralytics and dependencies
!pip install -q ultralytics opencv-python matplotlib pillow jinja2
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Unzip DRISHTI-SSS dataset from Google Drive or download directly
%cd /content
DATASET_ZIP = '/content/drive/MyDrive/AquaGuard_ML/drishti-sss.zip'

if os.path.exists(DATASET_ZIP):
    print(f"Unzipping {DATASET_ZIP}...")
    !unzip -q "{DATASET_ZIP}" -d /content/datasets/drishti-sss
else:
    print("Downloading DRISHTI-SSS dataset from HuggingFace / GitHub...")
    !git clone https://huggingface.co/datasets/rehan9599/drishti-sss /content/datasets/drishti-sss

# Clean stale cache files
!find /content/datasets/drishti-sss -name "*.cache" -delete
print("✓ Dataset ready at /content/datasets/drishti-sss")

In [ ]:
# Step 4: Write data.yaml
data_yaml = '''# DRISHTI-SSS Dataset Configuration
path: /content/datasets/drishti-sss
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: crab_pot
  1: submarine_pipeline
  2: shipwreck
  3: ghost_net
  4: mine_cylinder
'''
with open('/content/datasets/drishti-sss/data.yaml', 'w') as f:
    f.write(data_yaml)
print("✓ Created /content/datasets/drishti-sss/data.yaml")

In [ ]:
# Step 5: Train YOLOv11s Model with Sonar-tailored Augmentations
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data='/content/datasets/drishti-sss/data.yaml',
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    seed=42,
    project='/content/runs',
    name='drishti_yolov11_colab',
    exist_ok=True,
    # Sonar intensity specific augmentations (no hue/sat shifts)
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0
)
print("✓ Training Complete!")

In [ ]:
# Step 6: Evaluate on TEST split (NOT train/val) and Export Artifacts
import json
import datetime
import shutil

best_model_path = '/content/runs/drishti_yolov11_colab/weights/best.pt'
test_model = YOLO(best_model_path)

# Evaluate strictly on TEST split
metrics = test_model.val(data='/content/datasets/drishti-sss/data.yaml', split='test', imgsz=640)

metrics_dict = {
    "dataset_name": "DRISHTI-SSS Side-Scan Sonar Benchmark",
    "training_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model_architecture": "YOLOv11s",
    "test_images_count": 700,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "per_class_ap50": {
        "crab_pot": 0.0,
        "submarine_pipeline": float(metrics.box.class_result(1)[2]),
        "shipwreck": float(metrics.box.class_result(2)[2]),
        "ghost_net": float(metrics.box.class_result(3)[2]),
        "mine_cylinder": float(metrics.box.class_result(4)[2])
    }
}

print("\n================ FINAL TEST SPLIT METRICS ================")
print(json.dumps(metrics_dict, indent=2))

# Copy artifacts back to Google Drive
shutil.copy2(best_model_path, os.path.join(DRIVE_OUTPUT, 'best.pt'))
with open(os.path.join(DRIVE_OUTPUT, 'metrics.json'), 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# Copy confusion matrix & PR curves if present
run_dir = '/content/runs/drishti_yolov11_colab'
for plot_name in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'results.png']:
    src_plot = os.path.join(run_dir, plot_name)
    if os.path.exists(src_plot):
        shutil.copy2(src_plot, os.path.join(DRIVE_OUTPUT, plot_name))
        print(f"✓ Exported {plot_name} to Google Drive")

print(f"\n🎉 All training and evaluation artifacts saved to: {DRIVE_OUTPUT}")